In [18]:
import pandas as pd
import numpy as np


# A UDF that adds 1 to a value applied to a Series
def add_one(x):
    return x + 1


# Apply the function element-wise using .map
s = pd.Series([1, 2, 3])
s.map(add_one)

0    2
1    3
2    4
dtype: int64

# Why not to use User-Defined-Functions
UDFs are slower leading to inefficient processing.

> Use UDFs when custom computations are needed, for extending pandas functionality or for handling complex group operations.

In [19]:
from sklearn.linear_model import LinearRegression

# Sample data
df = pd.DataFrame(
    {
        "group": ["A", "A", "A", "B", "B", "B"],
        "x": [1, 2, 3, 1, 2, 3],
        "y": [2, 4, 6, 1, 2, 1.5],
    }
)


# Function to fit a model to each group
def fit_model(group):
    model = LinearRegression()
    model.fit(group[["x"]], group["y"])
    group["y_pred"] = model.predict(group[["x"]])

    return group


result = df.groupby("group").apply(fit_model)

In [20]:
result

x    y  y_pred
group                  
A     0  1  2.0    2.00
      1  2  4.0    4.00
      2  3  6.0    6.00
B     3  1  1.0    1.25
      4  2  2.0    1.50
      5  3  1.5    1.75

# Series.map() / DataFrame.map()

map() is executed **element-wise** i.e. for each element in the Series/DataFrame. 

The individual value of each cell is the argument of the UDF.

In [21]:
temperature_celsius = pd.DataFrame(
    {
        "NYC": [14, 21, 23],
        "Los Angeles": [22, 28, 31],
    }
)


def to_fahrenheit(value):
    return value * (9 / 5) + 32


temperature_celsius.map(to_fahrenheit)

,NYC,Los Angeles
0,57.2,71.6
1,69.8,82.4
2,73.4,87.8


# Series.apply() / DataFrame.apply()

apply() executes a UDF for a whole column or row. It is called for each row or column, *not* for each individual value.

The operations will be faster than map(), which is called on each element.

In [22]:
temperature_celsius = pd.DataFrame(
    {
        "NYC": [14, 21, 23],
        "Los Angeles": [22, 28, 31],
    }
)


def to_fahrenheit(column):
    return column * (9 / 5) + 32


temperature_celsius.apply(to_fahrenheit)

,NYC,Los Angeles
0,57.2,71.6
1,69.8,82.4
2,73.4,87.8


In [23]:
def normalize(column):
    return column / column.mean()


temperature_celsius.apply(normalize)

,NYC,Los Angeles
0,0.724138,0.814815
1,1.086207,1.037037
2,1.189655,1.148148


In [24]:
# apply() can also be executed for each row
def hotter(row):
    return row["Los Angeles"] - row["NYC"]


temperature_celsius.apply(hotter, axis=1)

0    8
1    7
2    8
dtype: int64

# Series.pipe() / DataFrame.pipe()

The UDF used in pipe() takes as argument the whole `Series` or `DataFrame`; not an individual value.

In [25]:
def normalize(df):
    return df / df.mean().mean()


temperature_celsius.pipe(normalize)

,NYC,Los Angeles
0,0.604317,0.949640
1,0.906475,1.208633
2,0.992806,1.338129


In [26]:
# The above is the same as calling the normalize() function with the DataFrame as argument:
normalize(temperature_celsius)

,NYC,Los Angeles
0,0.604317,0.949640
1,0.906475,1.208633
2,0.992806,1.338129


In [30]:
# pipe() allows method chaining
temperature_celsius = pd.DataFrame(
    {
        "NYC": [14, 21, 23],
        "Los Angeles": [22, 28, 31],
    }
)


def multiply_by_9(value):
    return value * 9


def divide_by_5(value):
    return value / 5


def add_32(value):
    return value + 32


# Without using pipe()
fahrenheit = add_32(divide_by_5(multiply_by_9(temperature_celsius)))

# With pipe()
fahrenheit = temperature_celsius.pipe(multiply_by_9).pipe(divide_by_5).pipe(add_32)

# Series.agg() and DataFrame.agg()

In [31]:
def highest_jump(column):
    return column.pct_change().max()


temperature_celsius.agg(highest_jump)

NYC            0.500000
Los Angeles    0.272727
dtype: float64

# Series.transform() and DataFrame.transform()

In [33]:
temperature_celsius = pd.DataFrame(
    {
        "NYC": [14, 21, 23],
        "Los Angeles": [22, 28, 31],
    },
    index=pd.date_range("2000-01-01", "2000-01-03"),
)


def warm_up_all_days(column):
    return pd.Series(column.max(), index=column.index)


temperature_celsius.transform(warm_up_all_days)

,NYC,Los Angeles
2000-01-01,23,31
2000-01-02,23,31
2000-01-03,23,31


# Performance
## Vectorized Operations

Vectorized operations in `pandas` are **significantly faster** than using DataFrame.apply() with UDFs because they are based on highly optimized C functions to process entire arrays at once.

In [34]:
df

,group,x,y
0,A,1,2.0
1,A,2,4.0
2,A,3,6.0
3,B,1,1.0
4,B,2,2.0
5,B,3,1.5


In [39]:
# User-defined function
def calc_ratio(row):
    return 100 * (row["x"] / row["y"])


%timeit df["new_col"] = df.apply(calc_ratio, axis=1)

180 μs ± 1.8 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [40]:
# Vectorized Operation
%timeit df["new_col2"] = 100 * df["x"] / df["y"]

107 μs ± 620 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
